In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib as m
from tqdm import tqdm
from scipy import linalg
import scipy as sp
import os
import pickle
import glob
import h5py
import random

from scipy.constants import *

phi0 = physical_constants["mag. flux quantum"][0]
eps0 = epsilon_0

from scipy.special import eval_hermite
import scfitpy as Qfit

In [ ]:
def Rabi(eps, params, F):
    bn = 4
    E1 = [[] for i in range(bn)]
    for e in eps:
        H = Qfit.QHami(F, e, params[0], params[1], params[2], 0, 0)
        evals, ekets = np.linalg.eigh(H)
        for q in range(bn):
            E1[q].append(evals[q])
    return np.array(E1)

In [ ]:
glist = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4]
er01 = [[] for i in range(len(glist))]
er20 = [[] for i in range(len(glist))]
er31 = [[] for i in range(len(glist))]
eps = [0]

for g in glist:
    for fock in range(2, 14):
        Rabi_params = [g, 0.2, 2, 0]
        e3 = Rabi(eps, Rabi_params, fock)
        er01[i].append(e3[1] - e3[0])
        er20[i].append(e3[2] - e3[0])
        er31[i].append(e3[3] - e3[1])

In [ ]:
import matplotlib.colors as mcolors
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import FormatStrFormatter
import matplotlib.cm as cm

custom_colors = ["black", "blue", "pink", "red", "grey", "green", "orange"]
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=custom_colors)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
plt.rcParams["font.size"] = 20
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "cm"
plt.rcParams["font.weight"] = 0

for i in range(len(er01)):
    plt.plot(np.linspace(2, 14, 12), er01[i] / er01[i][11], "*-", ms=10)
ax.set_xlim([1, 15])
ax.xaxis.set_major_locator(MaxNLocator(5))

plt.grid()
ax.set_ylabel(r"$\omega_{01}(F)/\omega_{01}(F=14)$", fontsize=20, fontweight=0)
ax.set_xlabel("Fock space", fontsize=20, fontweight=0)
fig.tight_layout()
# plt.savefig("Dw10Re0.png",bbox_inches="tight", pad_inches=0.1, dpi=500)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
plt.rcParams["font.size"] = 20
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "cm"
plt.rcParams["font.weight"] = 0

for i in range(len(er01)):
    plt.plot(np.linspace(2, 14, 12), er20[i] / er20[i][11], "*-", ms=10)
ax.set_xlim([1, 15])
ax.xaxis.set_major_locator(MaxNLocator(5))

plt.grid()
ax.set_ylabel(r"$\omega_{20}(F)/\omega_{20}(F=14)$", fontsize=20, fontweight=0)
ax.set_xlabel("Fock space", fontsize=20, fontweight=0)
fig.tight_layout()
# plt.savefig("Dw20Re0.png",bbox_inches="tight", pad_inches=0.1, dpi=500)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
plt.rcParams["font.size"] = 20
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "cm"
plt.rcParams["font.weight"] = 0

for i in range(len(er01)):
    plt.plot(np.linspace(2, 14, 12), er31[i] / er31[i][11], "*-", ms=10)
ax.set_xlim([1, 15])
ax.xaxis.set_major_locator(MaxNLocator(5))
ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))


plt.grid()
ax.set_ylabel(r"$\omega_{31}(F)/\omega_{31}(F=14)$", fontsize=20, fontweight=0)
ax.set_xlabel("Fock space", fontsize=20, fontweight=0)
fig.tight_layout()
# plt.savefig("Dw31Re0.png",bbox_inches="tight", pad_inches=0.1, dpi=500)
plt.show()

In [ ]:
f = open("PT_paper01.pickle", "rb")
Rabi_params = pickle.load(f)
er01_e = []
er20_e = []
er31_e = []
eps2 = np.linspace(0, 15, 101)

for fock in range(2, 15):
    e3 = Rabi(eps2, Rabi_params, fock)
    er01_e.append(e3[1] - e3[0])
    er20_e.append(e3[2] - e3[0])
    er31_e.append(e3[3] - e3[1])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
plt.rcParams["font.size"] = 20
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "stix"

for i in range(6):
    ax.plot(eps2, er01_e[i], color=custom_colors[i])
    ax.plot(eps2, er20_e[i], color=custom_colors[i])
    ax.plot(eps2, er31_e[i], color=custom_colors[i])
ax.set_ylim([5, 5.6])
ax.set_xlim([0, 15])
plt.grid()
plt.xlabel(r"$\varepsilon$[GHz]", fontsize=24)
plt.ylabel(r"$\omega_{ij}/2\pi$[GHz]", fontsize=22)
# plt.savefig("qspace_w31.png",dpi=700)
plt.show()

In [ ]:
resd = []
for i in range(len(er01_e)):
    resd.append(
        max(
            abs(
                er01_e[i] + er20_e[i] + er31_e[i] - er01_e[12] - er20_e[12] - er31_e[12]
            )
        )
    )

In [ ]:
from matplotlib.ticker import MultipleLocator, FormatStrFormatter

fig, ax = plt.subplots(figsize=(5, 4))
plt.rcParams["font.size"] = 18
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "stix"

ax.plot(np.linspace(2, 14, 13), np.array(resd), "-+")

ax.set_ylim([1e-8, 50])
ax.set_xlim([2, 13])
plt.xticks([3, 5, 7, 9, 11, 13])
plt.yscale("log")
plt.grid()
plt.xlabel(r"Fock space", fontsize=15)
plt.ylabel(r"Maximum energy difference [GHz]", fontsize=15)

# plt.savefig("Fspace.png",bbox_inches="tight", pad_inches=0.5,dpi=700)
plt.show()